# 07 - Extract

Plan section 39: freeze the evolved architecture and publish it through the checksummed manifest contract.

Reproducibility contract (plan section 40): every session begins by
determining its configuration, locating and validating prior state,
and continues from there; it ends by saving metrics and a summary JSON.
Run inside a Kaggle session (or locally) after checking out this
repository, with the repository root as the working directory.


In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd()
assert (REPO / 'src' / 'act').is_dir(), (
    f'run this notebook with the repository root as the working directory; '
    f'cwd was {REPO!r}'
)
sys.path.insert(0, str(REPO / 'src'))
print('python', sys.version.split()[0], '| repo', REPO)


# Freeze and extract (plan section 39)

The evolved architecture becomes a portable artifact: the genome
round-trips its stable hash through serialization, the versioned
checkpoint packages model + optimizer + genome + mutation history,
and the session publishes through the checksummed manifest
contract (plan section 40: save, then publish output).


In [ ]:
import torch
from act.topology import (
    ArchitectureGenome,
    create_genome,
    genome_hash,
    grow_expert,
)
from act.model import model_from_genome, rebuild_model_from_genome

genome = create_genome(42, hidden_size=24, vocab_size=32, depth=1,
                       ffn='moe', expert_count=4, active_experts=2)
model = model_from_genome(genome, context_size=12)
added = grow_expert(genome, genome.layers[0].id, step=12)
model = rebuild_model_from_genome(model, genome, context_size=12)
assert added is not None
hash_before = genome_hash(genome)
restored = ArchitectureGenome.from_dict(genome.to_dict())
assert genome_hash(restored) == hash_before, 'genome hash did not round trip'
print('evolved genome', hash_before, '| experts:',
      len(genome.layers[0].experts))

In [ ]:
# Freeze the full resumable state in the versioned checkpoint format.
from act.training.runtime import (
    TrainingConfig,
    TrainingState,
    load_checkpoint,
    make_optimizer,
    save_checkpoint,
)

CHECKPOINT = 'artifacts/notebook-records/07-extracted.pt'
optimizer = make_optimizer(model, TrainingConfig())
state = TrainingState(
    step=12,
    tokens=288,
    genome=genome.to_dict(),
    mutation_history=[
        {'step': 12, 'operation': 'grow', 'expert_id': added},
    ],
)
save_checkpoint(CHECKPOINT, model, optimizer, state)
payload = torch.load(CHECKPOINT, map_location='cpu', weights_only=False)
assert payload['version'] == 3
assert payload['training']['genome'] == genome.to_dict()
print('frozen', CHECKPOINT, '| genome + mutation history persisted')

In [ ]:
# Publish the session's artifacts behind the checksummed manifest.
import json
from pathlib import Path
from act.ops import kaggle_layout, publish_artifacts, restore_artifacts

report = Path('artifacts/notebook-records/07-extract-summary.json')
report.write_text(json.dumps({
    'architecture_hash': genome_hash(genome),
    'mutation_history': state.mutation_history,
}, sort_keys=True) + '\n')
layout = kaggle_layout('artifacts/notebook-records/07-kaggle')
manifest = publish_artifacts(
    layout,
    (CHECKPOINT, report),
    dataset_id='notebook-07-extract',
    code_revision='notebook',
    config_hash=genome_hash(genome),
)
assert Path(layout.manifests / 'manifest.json').is_file()
restored_files = restore_artifacts(layout)
assert len(restored_files) == 2
print('published:', manifest.artifacts)

In [ ]:
import json
from pathlib import Path

record_dir = Path('artifacts/notebook-records')
record_dir.mkdir(parents=True, exist_ok=True)
payload = {'notebook': '07_extract',
 'architecture_hash': genome_hash(genome),
 'checkpoint_version': payload['version'],
 'published': list(manifest.artifacts)}
record = json.dumps(payload, sort_keys=True, indent=2) + '\n'
target = record_dir / '07-extract.json'
existing = target.read_text() if target.exists() else None
if existing != record:
    target.write_text(record)
print('record:', target)